<a href="https://colab.research.google.com/github/ghroyd1110/Git-Commands/blob/Test/GeneratorExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 50.3 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import fitz # !pip install pymupdf
from google.colab import drive
from datetime import datetime

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# 2. CONFIGURATION
ROOT_FOLDER = '/content/drive/MyDrive/Permits'
LOG_FILE = '/content/drive/MyDrive/PermitsOutput/generator_data_log_20260330.txt'

def get_text_to_right(page, label, width=400):
    rects = page.search_for(label)
    if not rects: return "Not Found"
    target_rect = rects[0]
    search_zone = fitz.Rect(target_rect.x1, target_rect.y0 - 2, target_rect.x1 + width, target_rect.y1 + 2)
    return page.get_text("text", clip=search_zone).strip() or "Not Found"

def extract_adeq_universal(pdf_path):
    records = []
    try:
        doc = fitz.open(pdf_path)
        page1 = doc[0]

        p_no_raw = get_text_to_right(page1, "PERMIT No.")
        if p_no_raw == "Not Found": p_no_raw = get_text_to_right(page1, "PERMIT")
        permit_no = re.search(r"(\d+)", p_no_raw).group(1) if re.search(r"(\d+)", p_no_raw) else p_no_raw

        header_data = {
            "Permit No.": permit_no,
            "PERMITTEE": get_text_to_right(page1, "PERMITTEE:"),
            "FACILITY": get_text_to_right(page1, "FACILITY:"),
            "PLACE ID": get_text_to_right(page1, "PLACE ID:"),
            "DATE ISSUED": get_text_to_right(page1, "DATE ISSUED:"),
            "EXPIRY DATE": get_text_to_right(page1, "EXPIRY DATE:"),
            "Source File": os.path.basename(pdf_path)
        }

        for page in doc:
            words = page.get_text("words")
            words.sort(key=lambda w: (w[3], w[0]))

            lines = []
            if words:
                curr_y, curr_line = words[0][3], []
                for w in words:
                    if abs(w[3] - curr_y) < 5: curr_line.append(w[4])
                    else:
                        lines.append(" ".join(curr_line))
                        curr_line, curr_y = [w[4]], w[3]
                lines.append(" ".join(curr_line))

            for i, line in enumerate(lines):
                lower_line = line.lower()
                # Expanded keywords to catch the 45+ units in Yuma (104799)
                is_target = any(k in lower_line for k in ["generator", "engine", "fire pump", "hp", "kw"])

                if is_target and "equipment type" not in lower_line and "maximum capacity" not in lower_line:
                    try:
                        entry = header_data.copy()
                        # Capture the row and the next 7 data chunks for the "Raw Data" column
                        entry.update({
                            "Equipment Type": line,
                            "Raw Data Row": " | ".join(lines[i+1:i+8])
                        })
                        records.append(entry)
                    except: pass
        doc.close()
    except Exception as e:
        print(f"Error in {pdf_path}: {e}")
    return records

# 3. RUNTIME
print(f"Starting Universal Extraction at {datetime.now().strftime('%H:%M:%S')}...")

with open(LOG_FILE, 'a') as f:
    for root, dirs, files in os.walk(ROOT_FOLDER):
        for file in files:
            if file.lower().endswith(".pdf") and ("final" in file.lower() or "final" in root.lower()):
                results = extract_adeq_universal(os.path.join(root, file))
                for r in results:
                    f.write(json.dumps(r) + "\n")
                if results:
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] Logged {len(results)} units from {file}")

print(f"Finished! Log file updated at {LOG_FILE} on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Mounted at /content/drive
Starting Universal Extraction at 19:41:47...
[19:42:06] Logged 1 units from 220090_91800 Permit_Determination_Letter_signed.pdf
[19:42:09] Logged 2 units from Kirkland truck to rail loading final application packet_2011.11.19_Signed.pdf
[19:42:09] Logged 1 units from 220090_92044 NoPermitRequired_Letter(Final).pdf
[19:42:09] Logged 1 units from 220090_92044 NoPermitRequired_Letter(Final)_signed.pdf
[19:42:12] Logged 46 units from Final Permit.pdf
[19:42:12] Logged 12 units from Final TSD.pdf
[19:42:13] Logged 12 units from Final TSD-part 2.pdf
[19:42:17] Logged 34 units from 34918 Bowie Draft 02-17-06 R5.pdf
[19:42:59] Logged 125 units from FINAL BrightNight Class I Application Package_2025-1231 (1).pdf
[19:43:12] Logged 2 units from Issue Letter_56677 (Revised 3-13-15 per NRE request).pdf
[19:43:54] Logged 19 units from 40 CFR 63 subpart LLL_final rule FR.pdf
[19:43:56] Logged 2 units from 2869_86524 Issuance_Letter.pdf
[19:43:56] Logged 2 units from CalPortl

In [3]:
import pandas as pd
import json
import csv
import os
from google.colab import drive

# 1. MOUNT DRIVE (if not already mounted)
drive.mount('/content/drive')

# 2. CONFIGURATION - Update these to match your exact file names
LOG_FILE = '/content/drive/MyDrive/PermitsOutput/generator_data_log_20260330.txt'
FINAL_CSV = '/content/drive/MyDrive/PermitsOutput/Emergency_Generator_Final_Master_2_step_20260330.csv'

# 3. CONVERSION LOGIC
data = []

print(f"Reading data from {LOG_FILE}...")

if os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'r') as f:
        for line_num, line in enumerate(f, 1):
            try:
                # Load each line as a JSON object
                item = json.loads(line)
                data.append(item)
            except Exception as e:
                print(f"Skipping line {line_num} due to formatting error: {e}")

    if data:
        # Convert the list of dictionaries into a Table (DataFrame)
        df = pd.DataFrame(data)

        # CLEANUP: Remove exact duplicates (if the script was run twice on the same folder)
        # We define a duplicate as having the same Permit No and Equipment ID
        if "Permit No." in df.columns and "Equipment ID Number" in df.columns:
            initial_count = len(df)
            df = df.drop_duplicates(subset=["Permit No.", "Equipment ID Number"])
            print(f"Removed {initial_count - len(df)} duplicate entries.")

        # SAVE TO CSV WITH HEADERS AND DOUBLE QUOTES
        # quoting=csv.QUOTE_ALL ensures every field is wrapped in ""
        df.to_csv(FINAL_CSV, index=False, quoting=csv.QUOTE_ALL)

        print("-" * 30)
        print(f"SUCCESS!")
        print(f"Total Records Processed: {len(df)}")
        print(f"Final CSV Created: {FINAL_CSV}")
        print("-" * 30)
    else:
        print("The log file was empty. No data to convert.")
else:
    print(f"Error: Could not find {LOG_FILE}. Please check the file path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading data from /content/drive/MyDrive/PermitsOutput/generator_data_log_20260330.txt...
------------------------------
SUCCESS!
Total Records Processed: 113249
Final CSV Created: /content/drive/MyDrive/PermitsOutput/Emergency_Generator_Final_Master_2_step_20260330.csv
------------------------------
